# Ordenación rápida (quick sort)

Elige un pivote, coloca los menores a su izquierda y los mayores a su derecha (partición) y ordena recursivamente cada lado. Cuesta $O(n \log n)$ de media y $O(n^2)$ en el peor caso; un pivote aleatorio hace muy improbable el peor caso.

## 1. Quick sort sencillo

Versión funcional que crea listas nuevas en cada partición (fácil de leer, usa memoria extra).

In [1]:
import random

def quick_sort(lista):
    if len(lista) <= 1:
        return lista
    pivote = random.choice(lista)
    menores = [x for x in lista if x < pivote]
    iguales = [x for x in lista if x == pivote]
    mayores = [x for x in lista if x > pivote]
    return quick_sort(menores) + iguales + quick_sort(mayores)

print(quick_sort([10, 7, 8, 9, 1, 5, 7]))

[1, 5, 7, 7, 8, 9, 10]


## 2. Quick sort en el sitio

Implementar la partición de Lomuto, que reordena la lista sin crear listas nuevas.

In [2]:
def particion(l, izq, der):
    # pivote aleatorio al final
    p = random.randint(izq, der)
    l[p], l[der] = l[der], l[p]
    pivote, i = l[der], izq
    for j in range(izq, der):
        if l[j] < pivote:
            l[i], l[j] = l[j], l[i]
            i += 1
    l[i], l[der] = l[der], l[i]
    return i

def quick_sort_in_place(l, izq=0, der=None):
    if der is None:
        der = len(l) - 1
    if izq < der:
        p = particion(l, izq, der)
        quick_sort_in_place(l, izq, p - 1)
        quick_sort_in_place(l, p + 1, der)
    return l

for _ in range(200):
    datos = random.choices(range(100), k=random.randint(0, 50))
    assert quick_sort(datos) == sorted(datos)
    assert quick_sort_in_place(datos[:]) == sorted(datos)
print(quick_sort_in_place([10, 7, 8, 9, 1, 5, 7]))

[1, 5, 7, 7, 8, 9, 10]


## 3. Partición en 3 vías

Con muchos valores repetidos, la partición de Lomuto deja todos los iguales al pivote en el mismo lado y la recursión degenera. Implementar la partición en 3 vías (la bandera holandesa de Dijkstra), que separa menores, iguales y mayores que el pivote, y ordenar recursivamente solo menores y mayores. Probarla con 5000 valores entre 0 y 2.

In [3]:
def particion_3_vias(l, izq, der):
    pivote = l[random.randint(izq, der)]
    menor, i, mayor = izq, izq, der
    # [izq, menor) < pivote, [menor, i) == pivote, (mayor, der] > pivote
    while i <= mayor:
        if l[i] < pivote:
            l[menor], l[i] = l[i], l[menor]
            menor += 1
            i += 1
        elif l[i] > pivote:
            l[i], l[mayor] = l[mayor], l[i]
            mayor -= 1
        else:
            i += 1
    return menor, mayor

def quick_sort_3_vias(l, izq=0, der=None):
    if der is None:
        der = len(l) - 1
    if izq < der:
        menor, mayor = particion_3_vias(l, izq, der)
        quick_sort_3_vias(l, izq, menor - 1)
        quick_sort_3_vias(l, mayor + 1, der)
    return l

random.seed(3)
for _ in range(300):
    datos = random.choices(range(10), k=random.randint(0, 50))
    assert quick_sort_3_vias(datos[:]) == sorted(datos)
print(quick_sort_3_vias([3, 1, 2, 3, 3, 1, 2, 2, 3, 1]))

repetidos = random.choices(range(3), k=5000)
try:
    quick_sort_in_place(repetidos[:])
except RecursionError as e:
    print("Lomuto:", type(e).__name__, "-", e)
print("3 vías ordena bien:", quick_sort_3_vias(repetidos[:]) == sorted(repetidos))

[1, 1, 1, 2, 2, 2, 3, 3, 3, 3]
Lomuto: RecursionError - maximum recursion depth exceeded
3 vías ordena bien: True


## 4. Mejor y peor caso según el pivote

Contar comparaciones con dos estrategias de pivote: el último elemento y uno aleatorio. Con pivote fijo, una lista ya ordenada (o inversa) es el peor caso: cada partición solo separa un elemento y se hacen $\frac{n(n-1)}{2}$ comparaciones. Con pivote aleatorio el número esperado es $2n \ln n \approx 1{,}39\, n \log_2 n$ para cualquier entrada. Usar una pila de tramos pendientes para que el peor caso no agote la recursión.

In [4]:
import math

def quick_sort_contando(lista, pivote_aleatorio):
    l, comparaciones, pendientes = lista[:], 0, [(0, len(lista) - 1)]
    while pendientes:
        izq, der = pendientes.pop()
        if izq >= der:
            continue
        if pivote_aleatorio:
            p = random.randint(izq, der)
            l[p], l[der] = l[der], l[p]
        pivote, i = l[der], izq
        for j in range(izq, der):
            comparaciones += 1
            if l[j] < pivote:
                l[i], l[j] = l[j], l[i]
                i += 1
        l[i], l[der] = l[der], l[i]
        pendientes += [(izq, i - 1), (i + 1, der)]
    return l, comparaciones

random.seed(4)
n = 1000
casos = {"ordenada": list(range(n)), "inversa": list(range(n, 0, -1)), "aleatoria": random.sample(range(n), n)}
print(f"{'caso':>10} {'pivote último':>14} {'pivote aleatorio':>17}")
for nombre, datos in casos.items():
    l_fijo, c_fijo = quick_sort_contando(datos, False)
    l_azar, c_azar = quick_sort_contando(datos, True)
    assert l_fijo == l_azar == sorted(datos)
    print(f"{nombre:>10} {c_fijo:>14} {c_azar:>17}")
    if nombre != "aleatoria":
        assert c_fijo == n * (n - 1) // 2
print(f"Referencia: n(n-1)/2 = {n * (n - 1) // 2}, 2n ln n ≈ {2 * n * math.log(n):.0f}")

      caso  pivote último  pivote aleatorio
  ordenada         499500             12898
   inversa         499500             11512
 aleatoria          10228             11027
Referencia: n(n-1)/2 = 499500, 2n ln n ≈ 13816
